# RAG Sprint 1 – מנוע חיפוש סמנטי על "מדריך לרוכש דירה"

נוטבוק זה בונה את שכבת ה־**Retrieval** של מערכת RAG, שלב אחר שלב:

1. **טעינת המסמך** – קריאת ה־PDF והמרתו לטקסט.
2. **חלוקה ל־Chunks** – פירוק הטקסט לקטעים קטנים וחופפים.
3. **Embeddings** – המרת כל chunk לוקטור מספרים באמצעות `gemini-embedding-001`.
4. **Pinecone** – שמירת הוקטורים במסד נתונים וקטורי.
5. **Semantic Search** – חיפוש לפי משמעות + הצגת התוצאות הרלוונטיות.

> כל שלב מתועד בתא Markdown שמסביר מה קורה ולמה.

## שלב 2 – טעינת המסמך וחלוקה ל־Chunks

**למה מחלקים ל־Chunks?**
מודל ה־Embeddings וה־Retrieval עובדים טוב יותר על קטעים קצרים וממוקדים מאשר על מסמך שלם.
בנוסף, יש הגבלת אורך לקלט. לכן מפרקים את הטקסט לקטעים ("chunks").

**מה זה `chunk_overlap`?**
כדי לא "לחתוך" משפט או רעיון באמצע בין שני chunks, אנחנו משאירים חפיפה של כמה תווים
בין chunk לחבירו. כך מידע שנמצא על הגבול עדיין מופיע בשלמותו באחד הקטעים.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pypdf import PdfReader

# --- טעינת מפתחות ה-API מקובץ .env ---
# load_dotenv מחפש את הקובץ .env ומכניס את המשתנים ל-os.environ.
# find .env בתיקיית השורש של הפרויקט (רמה אחת מעל notebooks/).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# בדיקה שהמפתחות נטענו (בלי להדפיס אותם!)
print("GEMINI_API_KEY נטען:", "כן" if GEMINI_API_KEY else "לא - בדקי את קובץ .env")
print("PINECONE_API_KEY נטען:", "כן" if PINECONE_API_KEY else "לא - בדקי את קובץ .env")

# --- הגדרות מרכזיות (נשנה אותן בשלב הבונוס) ---
PDF_PATH = PROJECT_ROOT / "data" / "apartment_buyer_guide.pdf"
CHUNK_SIZE = 1000       # אורך כל chunk בתווים
CHUNK_OVERLAP = 150     # חפיפה בתווים בין chunks עוקבים

EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768         # מימד הוקטור (חייב להתאים למימד ה-index ב-Pinecone)
INDEX_NAME = "rag-apartment-guide"

print("\nמסמך:", PDF_PATH)
print("קיים:", PDF_PATH.exists())

In [ ]:
import re


def clean_text(text):
    """ניקוי טקסט שחולץ מ-PDF: מכווץ רצפים ארוכים של רווחים ושורות ריקות
    (ה-PDF מכיל הרבה שורות ריקות שמוסיפות רעש ל-Embeddings)."""
    text = re.sub(r"[ \t]+", " ", text)      # רצף רווחים -> רווח בודד
    text = re.sub(r"\n\s*\n+", "\n", text)   # שורות ריקות מרובות -> שורה אחת
    return text.strip()


def load_pdf(pdf_path):
    """קורא PDF ומחזיר:
    - full_text: כל הטקסט של המסמך כמחרוזת אחת (מנוקה)
    - char_page: רשימה שממפה כל תו במיקום i למספר העמוד שממנו הגיע (לצורך metadata)
    """
    reader = PdfReader(str(pdf_path))
    full_text = ""
    char_page = []
    for page_num, page in enumerate(reader.pages, start=1):
        page_text = clean_text(page.extract_text() or "") + "\n"
        full_text += page_text
        char_page.extend([page_num] * len(page_text))
    return full_text, char_page


full_text, char_page = load_pdf(PDF_PATH)

print(f"מספר עמודים: {char_page[-1] if char_page else 0}")
print(f"סך תווים בטקסט: {len(full_text):,}")
print("\n--- תצוגה מקדימה (300 תווים ראשונים) ---")
print(full_text[:300])

In [ ]:
def chunk_text(text, char_page, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """מחלק טקסט ארוך לקטעים בגודל chunk_size תווים, עם חפיפה של chunk_overlap.
    כל chunk מקבל metadata עם מספר העמוד שבו הוא מתחיל.
    מחזיר רשימת dict: {id, text, page}.
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap חייב להיות קטן מ-chunk_size")

    chunks = []
    step = chunk_size - chunk_overlap  # כמה מתקדמים כל פעם
    idx = 0
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_str = text[start:end].strip()
        if chunk_str:  # מדלגים על קטעים ריקים
            page = char_page[start] if start < len(char_page) else char_page[-1]
            chunks.append({
                "id": f"chunk-{idx}",
                "text": chunk_str,
                "page": page,
            })
            idx += 1
        start += step
    return chunks

In [ ]:
chunks = chunk_text(full_text, char_page, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"נוצרו {len(chunks)} chunks (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
print("\n--- דוגמה: ה-chunk הראשון ---")
print("id:", chunks[0]["id"], "| עמוד:", chunks[0]["page"])
print(chunks[0]["text"][:400], "...")

print("\n--- דוגמה: chunk מאמצע המסמך ---")
mid = len(chunks) // 2
print("id:", chunks[mid]["id"], "| עמוד:", chunks[mid]["page"])
print(chunks[mid]["text"][:400], "...")

## שלב 3 – יצירת Embeddings עם Gemini

**מה זה Embedding?**
Embedding הוא ייצוג מספרי (וקטור) של טקסט. טקסטים עם משמעות דומה מקבלים וקטורים "קרובים" במרחב.

**למה `task_type`?**
למודל `gemini-embedding-001` מומלץ להפריד:
- `RETRIEVAL_DOCUMENT` – לטקסטים שנשמרים בבסיס הידע (chunks)
- `RETRIEVAL_QUERY` – לשאלות של המשתמש (נשתמש בזה בשלב החיפוש)

**אם מקבלים שגיאת SSL (נפוץ ב-NetFree):**
1. ודאי שקיים קובץ `.env` (לא `.env.example`) עם המפתחות האמיתיים.
2. התקיני את תעודת השורש של NetFree במחשב.
3. אם עדיין נכשל – הריצי בטרמינל: `pip install pip-system-certs` ואז **Restart Kernel** בנוטבוק.

In [ ]:
from google import genai
from google.genai import types

# יצירת לקוח Gemini עם המפתח מ-.env
# (המפתח נטען בתא ההגדרות למעלה)
if not GEMINI_API_KEY:
    raise ValueError(
        "חסר GEMINI_API_KEY. צרי קובץ .env בשורש הפרויקט:\n"
        "Copy-Item .env.example .env\n"
        "ואז מלאי את המפתח האמיתי ב-.env (לא ב-.env.example!)"
    )

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client נוצר בהצלחה")

In [ ]:
def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """ממיר רשימת טקסטים לוקטורים באמצעות gemini-embedding-001.
    מחזיר רשימת וקטורים (list of lists).
    """
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBED_DIM,
        ),
    )
    return [emb.values for emb in result.embeddings]


# בדיקה על chunk אחד
sample_vector = embed_texts([chunks[0]["text"]], task_type="RETRIEVAL_DOCUMENT")[0]

print(f"מימד הוקטור: {len(sample_vector)}")
print(f"3 ערכים ראשונים: {sample_vector[:3]}")
print(f"3 ערכים אחרונים: {sample_vector[-3:]}")

In [ ]:
# יצירת embeddings לכל ה-chunks (זה לוקח כמה שניות)
all_texts = [c["text"] for c in chunks]
all_vectors = embed_texts(all_texts, task_type="RETRIEVAL_DOCUMENT")

# שיוך הוקטור לכל chunk
for chunk, vector in zip(chunks, all_vectors):
    chunk["vector"] = vector

print(f"נוצרו embeddings ל-{len(all_vectors)} chunks")
print(f"דוגמה: {chunks[0]['id']} -> וקטור באורך {len(chunks[0]['vector'])}")

## שלב 4 – שמירה ב-Pinecone (Vector Database)

**מה זה Pinecone?**
מסד נתונים וקטורי שיודע לחפש במהירות "איזה וקטורים הכי דומים" לשאלה.
ב-RAG, אחרי שיצרנו embeddings לכל chunk – שומרים אותם ב-Pinecone עם metadata (טקסט, עמוד).

**מה קורה כאן?**
1. מתחברים ל-Pinecone עם המפתח מ-`.env`
2. יוצרים index (אם עדיין לא קיים) עם מימד 768 ו-metric `cosine`
3. מעלים (upsert) את כל ה-chunks עם הוקטורים שלהם

In [ ]:
from pinecone import Pinecone, ServerlessSpec

if not PINECONE_API_KEY:
    raise ValueError(
        "חסר PINECONE_API_KEY. מלאי אותו בקובץ .env (לא ב-.env.example)."
    )

pc = Pinecone(api_key=PINECONE_API_KEY)

# יצירת index רק אם הוא לא קיים
existing_indexes = {idx.name for idx in pc.list_indexes()}
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"נוצר index חדש: {INDEX_NAME}")
else:
    print(f"Index '{INDEX_NAME}' כבר קיים – משתמשים בו")

index = pc.Index(INDEX_NAME)
print("מחוברים ל-Pinecone index:", INDEX_NAME)

In [ ]:
# הכנת records ל-upsert: (id, vector, metadata)
records = [
    (
        chunk["id"],
        chunk["vector"],
        {
            "text": chunk["text"][:1000],  # Pinecone מגביל metadata – שומרים תחילת הטקסט
            "page": chunk["page"],
        },
    )
    for chunk in chunks
]

# upsert = הוספה/עדכון וקטורים ב-index
index.upsert(vectors=records)

stats = index.describe_index_stats()
print(f"הועלו {len(records)} vectors")
print("סטטיסטיקות index:", stats)

## שלב 5 – Semantic Search (חיפוש סמנטי)

**איך זה עובד?**
1. לוקחים שאלה של המשתמש
2. יוצרים embedding לשאלה עם `RETRIEVAL_QUERY`
3. שולחים את הוקטור ל-Pinecone ומבקשים את 3 התוצאות הקרובות ביותר (`top_k=3`)
4. בודקים אם התוצאות באמת רלוונטיות לשאלה

In [ ]:
def semantic_search(question, top_k=3):
    """מחפש את top_k הקטעים הרלוונטיים ביותר לשאלה."""
    query_vector = embed_texts([question], task_type="RETRIEVAL_QUERY")[0]
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    return results.matches


def print_results(question, matches):
    print("=" * 70)
    print("שאלה:", question)
    print("=" * 70)
    for i, match in enumerate(matches, start=1):
        page = match.metadata.get("page", "?")
        text = match.metadata.get("text", "")
        print(f"\n#{i} | score: {match.score:.4f} | page: {page} | id: {match.id}")
        print(text[:350], "...")

In [ ]:
# 5 שאלות לדוגמה על המדריך
questions = [
    "מה צריך לבדוק לגבי היתר בנייה לפני רכישת דירה?",
    "איך מבטיחים את כספי הרוכש לפי חוק המכר?",
    "מה קורה אם המוכר מאחר במסירת הדירה?",
    "מה לבדוק בחוזה המכר לפני חתימה?",
    "מהי תקופת הבדק ומה אחריות המוכר?",
]

for q in questions:
    matches = semantic_search(q, top_k=3)
    print_results(q, matches)
    print()